# Ring attention

The numerical pieces of causal ring attention, without communication.

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)

## Safe softmax

Subtracting the maximum does not change softmax, but prevents overflow.

In [ ]:
logits = torch.tensor([1000.0, 1001.0, 1002.0])

# Compute without using safe softmax -> will lead to infinity or NaN
unsafe = torch.exp(logits) / torch.exp(logits).sum()

# Compute by using safe softmax
maximum = logits.max()
safe = torch.exp(logits - maximum)
safe = safe / safe.sum()

print("unsafe:", unsafe)
print("safe:  ", safe)

# The two computations produce the same result
torch.testing.assert_close(safe, torch.softmax(logits, dim=0))

## Online softmax

Each block has a maximum $m_b$ and shifted exponential sum $l_b$. When a block arrives:

$$m_{new} = \max(m, m_b)$$

$$l_{new} = e^{m-m_{new}}l + e^{m_b-m_{new}}l_b$$

In [ ]:
# This is the original vector to which we want to apply the softmax
logits = torch.tensor([2.0, -1.0, 3.0, 0.5, 8.0, 7.0, -4.0, 1.0])
# Each block is made of 2 values
blocks = logits.chunk(4)

running_max = torch.tensor(float("-inf"))
# running_sum = "l" in the expression above
running_sum = torch.tensor(0.0)

# SCENARIO: we pretend we are the rank 0, which initially has the block 0 and sequentially asks for the other ranks for their own blocks

for block_index, block in enumerate(blocks):
    block_max = block.max()
    # block_sum = "l_b" in the expression above
    # We are computing the sum of the block, which may not be the global sum
    block_sum = torch.exp(block - block_max).sum()
    # ... but that's fine, because we keep track of the max sum seen so far.
    new_max = torch.maximum(running_max, block_max)

    running_sum = (
        torch.exp(running_max - new_max) * running_sum
        + torch.exp(block_max - new_max) * block_sum
    )
    running_max = new_max

    print(f"block {block_index}: max={running_max:.1f}, sum={running_sum:.4f}")

probabilities = torch.cat(
    [torch.exp(block - running_max) / running_sum for block in blocks]
)

reference_probabilities = torch.softmax(logits, dim=0)
torch.testing.assert_close(probabilities, reference_probabilities)
print("probabilities:\t\t\t", probabilities)
print("reference probabilities:\t", reference_probabilities)

## Causal attention in four blocks

$$\mathrm{softmax}(QK^T / \sqrt{d})V$$

Each query block attends to earlier KV blocks and its triangularly masked local KV block.

In [ ]:
sequence_length = 8
block_size = 2
head_dim = 4

generator = torch.Generator().manual_seed(0)
q = torch.randn(sequence_length, head_dim, generator=generator)
k = torch.randn(sequence_length, head_dim, generator=generator)
v = torch.randn(sequence_length, head_dim, generator=generator)

scale = head_dim**-0.5
scores = q @ k.T * scale
causal_mask = torch.ones(sequence_length, sequence_length, dtype=torch.bool).tril()
scores = scores.masked_fill(~causal_mask, float("-inf"))
reference = torch.softmax(scores, dim=-1) @ v

In [ ]:
# The number of blocks is sequence_length // block_size
q_blocks = q.split(block_size, dim=0)
k_blocks = k.split(block_size, dim=0)
v_blocks = v.split(block_size, dim=0)
local_mask = torch.ones(block_size, block_size, dtype=torch.bool).tril()


def make_state():
    return {
        "running_max": torch.full((block_size,), float("-inf")),
        "running_sum": torch.zeros(block_size),
        "running_values": torch.zeros(block_size, head_dim),
    }


def receive_kv_block(
    q_block,
    k_block,
    v_block,
    state,
    is_local=False,
    compute=True,
):
    if not compute:
        return state["running_values"] / state["running_sum"].unsqueeze(-1)

    block_scores = q_block @ k_block.T * scale
    if is_local:
        block_scores = block_scores.masked_fill(~local_mask, float("-inf"))

    block_max = block_scores.amax(dim=-1)
    new_max = torch.maximum(state["running_max"], block_max)
    previous_scale = torch.exp(state["running_max"] - new_max)
    block_weights = torch.exp(block_scores - new_max.unsqueeze(-1))

    state["running_sum"] = (
        previous_scale * state["running_sum"] + block_weights.sum(dim=-1)
    )
    state["running_values"] = (
        previous_scale.unsqueeze(-1) * state["running_values"]
        + block_weights @ v_block
    )
    state["running_max"] = new_max

    # The last call for each rank will return the correct result, ignore all intermediate ones.
    return state["running_values"] / state["running_sum"].unsqueeze(-1)


rank_states = [make_state(), make_state(), make_state(), make_state()]
attention_output = [
    torch.empty_like(q_blocks[0]),
    torch.empty_like(q_blocks[1]),
    torch.empty_like(q_blocks[2]),
    torch.empty_like(q_blocks[3]),
]

# Step 0: owners [0, 1, 2, 3]
attention_output[0] = receive_kv_block(
    q_blocks[0], k_blocks[0], v_blocks[0], rank_states[0], is_local=True
)
attention_output[1] = receive_kv_block(
    q_blocks[1], k_blocks[1], v_blocks[1], rank_states[1], is_local=True
)
attention_output[2] = receive_kv_block(
    q_blocks[2], k_blocks[2], v_blocks[2], rank_states[2], is_local=True
)
attention_output[3] = receive_kv_block(
    q_blocks[3], k_blocks[3], v_blocks[3], rank_states[3], is_local=True
)

# Step 1: owners [3, 0, 1, 2]
attention_output[0] = receive_kv_block(
    q_blocks[0], k_blocks[3], v_blocks[3], rank_states[0], compute=False
)
attention_output[1] = receive_kv_block(
    q_blocks[1], k_blocks[0], v_blocks[0], rank_states[1]
)
attention_output[2] = receive_kv_block(
    q_blocks[2], k_blocks[1], v_blocks[1], rank_states[2]
)
attention_output[3] = receive_kv_block(
    q_blocks[3], k_blocks[2], v_blocks[2], rank_states[3]
)

# Step 2: owners [2, 3, 0, 1]
attention_output[0] = receive_kv_block(
    q_blocks[0], k_blocks[2], v_blocks[2], rank_states[0], compute=False
)
attention_output[1] = receive_kv_block(
    q_blocks[1], k_blocks[3], v_blocks[3], rank_states[1], compute=False
)
attention_output[2] = receive_kv_block(
    q_blocks[2], k_blocks[0], v_blocks[0], rank_states[2]
)
attention_output[3] = receive_kv_block(
    q_blocks[3], k_blocks[1], v_blocks[1], rank_states[3]
)

# Step 3: owners [1, 2, 3, 0]
attention_output[0] = receive_kv_block(
    q_blocks[0], k_blocks[1], v_blocks[1], rank_states[0], compute=False
)
attention_output[1] = receive_kv_block(
    q_blocks[1], k_blocks[2], v_blocks[2], rank_states[1], compute=False
)
attention_output[2] = receive_kv_block(
    q_blocks[2], k_blocks[3], v_blocks[3], rank_states[2], compute=False
)
attention_output[3] = receive_kv_block(
    q_blocks[3], k_blocks[0], v_blocks[0], rank_states[3]
)

blockwise = torch.cat(attention_output)
torch.testing.assert_close(blockwise, reference, atol=1e-5, rtol=1e-5)
print("PASS: four-rank causal attention matches full attention")